# 📌 NOTE CHỤP HÌNH CHO BÁO CÁO
Các ghi chú có biểu tượng **📸 CHÈN HÌNH BÁO CÁO** đã được đặt trực tiếp ngay trên cell cần chụp. Dùng `Ctrl+F` và tìm cụm **CHÈN HÌNH BÁO CÁO** để nhảy nhanh giữa các vị trí.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt

In [2]:
X_train = pd.read_csv('./data/preprocessed-data/X_train_clean.csv')
X_test = pd.read_csv('./data/preprocessed-data/X_test_clean.csv')
y_train = pd.read_csv('./data/preprocessed-data/y_train.csv').squeeze() # .squeeze() để biến DataFrame 1 cột thành Series
y_test = pd.read_csv('./data/preprocessed-data/y_test.csv').squeeze()

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Date              800 non-null    object
 1   Gender            800 non-null    object
 2   Age               800 non-null    int64 
 3   Product Category  800 non-null    object
 4   Quantity          800 non-null    int64 
 5   Price per Unit    800 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 37.6+ KB


# Feature Creation

---
> 📸 **HÌNH 20 — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-01] — Mục 4.3 – Feature Creation**  
> Chụp cell code ngay bên dưới: hàm tạo Month, DayOfWeek, Is_Weekend và Gender_x_Category. Có thể chụp thêm output X_train_fe.head() ở cell kế tiếp.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [4]:
def create_features(data):
    df_fe = data.copy()
    
    df_fe['Date'] = pd.to_datetime(df_fe['Date'])
    df_fe['Month'] = df_fe['Date'].dt.month
    df_fe['DayOfWeek'] = df_fe['Date'].dt.dayofweek 
    
    df_fe['Is_Weekend'] = df_fe['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    
    df_fe['Gender_x_Category'] = df_fe['Gender'].astype(str) + "_" + df_fe['Product Category'].astype(str)
    
    return df_fe

In [5]:
X_train_fe = create_features(X_train)
X_test_fe = create_features(X_test)

# Feature Selection

---
> 📸 **HÌNH 21A (TÙY CHỌN) — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-02] — Mục 4.4 – Feature Extraction/Selection**  
> Chụp cell code ngay bên dưới để minh họa các cột bị loại và các đặc trưng được giữ lại.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [6]:
cols_to_drop = ['Date', 'Gender', 'Product Category', 'Quantity']

X_train_sel = X_train_fe.drop(columns=[c for c in cols_to_drop if c in X_train_fe.columns])
X_test_sel = X_test_fe.drop(columns=[c for c in cols_to_drop if c in X_test_fe.columns])

# Encoding Categorical Variables

In [7]:
ohe_cols = ['Gender_x_Category']
num_cols = ['Age', 'Month', 'DayOfWeek', 'Price per Unit']
pass_cols = ['Is_Weekend']

---
> 📸 **HÌNH 19 — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-03] — Mục 4.2.2 – One-Hot Encoding**  
> Chụp cell code ngay bên dưới, gồm OneHotEncoder, fit_transform/transform và get_feature_names_out().  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [8]:
# One Hot Encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_train_matrix = ohe.fit_transform(X_train_sel[ohe_cols])
ohe_feature_names = ohe.get_feature_names_out(ohe_cols)
df_ohe_train = pd.DataFrame(ohe_train_matrix, columns=ohe_feature_names, index=X_train_sel.index)

ohe_test_matrix = ohe.transform(X_test_sel[ohe_cols])
df_ohe_test = pd.DataFrame(ohe_test_matrix, columns=ohe_feature_names, index=X_test_sel.index)

print('Các cột sau One-Hot Encoding:')
print(list(ohe_feature_names))
print(f'Shape OHE train/test: {df_ohe_train.shape} / {df_ohe_test.shape}')
display(df_ohe_train.head())

Các cột sau One-Hot Encoding:
['Gender_x_Category_Female_Beauty', 'Gender_x_Category_Female_Clothing', 'Gender_x_Category_Female_Electronics', 'Gender_x_Category_Male_Beauty', 'Gender_x_Category_Male_Clothing', 'Gender_x_Category_Male_Electronics']
Shape OHE train/test: (800, 6) / (200, 6)


,Gender_x_Category_Female_Beauty,Gender_x_Category_Female_Clothing,Gender_x_Category_Female_Electronics,Gender_x_Category_Male_Beauty,Gender_x_Category_Male_Clothing,Gender_x_Category_Male_Electronics
0,1.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0


---
> 📸 **HÌNH 18 — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-04] — Mục 4.1.2 – Log Transform**  
> Chụp cell code ngay bên dưới, gồm np.log1p(y_train) và np.log1p(y_test).  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [9]:
# Log-Transform
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f'Skewness trước log: {y_train.skew():.4f}')
print(f'Skewness sau log1p: {pd.Series(y_train_log).skew():.4f}')
print(pd.DataFrame({'Total Amount': y_train.head(), 'log1p(Total Amount)': pd.Series(y_train_log).head()}))

Skewness trước log: 1.3969
Skewness sau log1p: 0.2834
   Total Amount  log1p(Total Amount)
0           900             6.803505
1           120             4.795791
2           200             5.303305
3            25             3.258097
4            90             4.510860


---
> 📸 **HÌNH 17 — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-05] — Mục 4.1.1 – Feature Scaling**  
> Chụp cell code ngay bên dưới, gồm StandardScaler, fit_transform trên train và transform trên test.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [10]:
# Feature Scaling
scaler = StandardScaler()

num_train_matrix = scaler.fit_transform(X_train_sel[num_cols])
df_num_train = pd.DataFrame(num_train_matrix, columns=num_cols, index=X_train_sel.index)

num_test_matrix = scaler.transform(X_test_sel[num_cols])
df_num_test = pd.DataFrame(num_test_matrix, columns=num_cols, index=X_test_sel.index)

print('Thống kê StandardScaler trên tập train:')
scaler_stats = pd.DataFrame({'Feature': num_cols, 'Mean': scaler.mean_, 'Scale': scaler.scale_})
display(scaler_stats.round(4))
display(df_num_train.head().round(4))

Thống kê StandardScaler trên tập train:


,Feature,Mean,Scale
0,Age,41.8225,13.6147
1,Month,6.5850,3.4584
2,DayOfWeek,3.0188,2.0170
3,Price per Unit,178.1000,189.7374


,Age,Month,DayOfWeek,Price per Unit
0,-0.2073,0.9875,1.4780,0.6425
1,0.9679,-1.0366,1.4780,-0.7806
2,0.6006,0.6983,-0.5051,-0.6751
3,-0.0604,0.9875,1.4780,-0.8069
4,-1.7498,0.1200,0.9823,-0.7806


In [11]:
X_train_final = pd.concat([df_num_train, df_ohe_train, X_train_sel[pass_cols]], axis=1)
X_test_final = pd.concat([df_num_test, df_ohe_test, X_test_sel[pass_cols]], axis=1)

print('Danh sách 11 đặc trưng cuối cùng:')
print(list(X_train_final.columns))
display(X_train_final.head().round(4))


Danh sách 11 đặc trưng cuối cùng:
['Age', 'Month', 'DayOfWeek', 'Price per Unit', 'Gender_x_Category_Female_Beauty', 'Gender_x_Category_Female_Clothing', 'Gender_x_Category_Female_Electronics', 'Gender_x_Category_Male_Beauty', 'Gender_x_Category_Male_Clothing', 'Gender_x_Category_Male_Electronics', 'Is_Weekend']


,Age,Month,DayOfWeek,Price per Unit,Gender_x_Category_Female_Beauty,Gender_x_Category_Female_Clothing,Gender_x_Category_Female_Electronics,Gender_x_Category_Male_Beauty,Gender_x_Category_Male_Clothing,Gender_x_Category_Male_Electronics,Is_Weekend
0,-0.2073,0.9875,1.4780,0.6425,1.0,0.0,0.0,0.0,0.0,0.0,1
1,0.9679,-1.0366,1.4780,-0.7806,1.0,0.0,0.0,0.0,0.0,0.0,1
2,0.6006,0.6983,-0.5051,-0.6751,0.0,1.0,0.0,0.0,0.0,0.0,0
3,-0.0604,0.9875,1.4780,-0.8069,0.0,1.0,0.0,0.0,0.0,0.0,1
4,-1.7498,0.1200,0.9823,-0.7806,0.0,0.0,0.0,1.0,0.0,0.0,1


---
> 📸 **HÌNH 21 — CHÈN HÌNH BÁO CÁO [03-FEATURE-ENGINEERING-06] — Mục 4.4 – Kích thước dữ liệu sau Feature Engineering**  
> Chụp cả cell code và output ngay bên dưới. Output đúng mong đợi: X_train_final (800, 11), X_test_final (200, 11).  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [12]:
print(f"Kích thước X_train_final hoàn chỉnh: {X_train_final.shape}")
print(f"Kích thước X_test_final hoàn chỉnh: {X_test_final.shape}")

Kích thước X_train_final hoàn chỉnh: (800, 11)
Kích thước X_test_final hoàn chỉnh: (200, 11)


In [13]:
os.makedirs('./data/ready_for_train', exist_ok=True)

joblib.dump(X_train_final, './data/ready_for_train/X_train_final.pkl')
joblib.dump(X_test_final, './data/ready_for_train/X_test_final.pkl')

joblib.dump(y_train_log, './data/ready_for_train/y_train_log.pkl')
joblib.dump(y_test_log, './data/ready_for_train/y_test_log.pkl')

# Dành cho Model Deployment
joblib.dump(scaler, './data/ready_for_train/custom_scaler.pkl')
joblib.dump(ohe, './data/ready_for_train/custom_ohe.pkl')

['./data/ready_for_train/custom_ohe.pkl']